[← 05 - Window Functions, Part 1](<05 - Window Functions, Part 1.ipynb>) · [Course Overview](<00 - Course Overview.ipynb>)

# 06 - Window Functions, Part 2

`05 - Window Functions, Part 1` covered `ROW_NUMBER()`, `LAG()`/`LEAD()`, and a running total with the default frame. This notebook covers the part most people skip: the frame clause itself, `NTILE()` for splitting rows into buckets, `PERCENT_RANK()` and `CUME_DIST()` for relative standing, and a specific, well-known trap with `LAST_VALUE()` that catches people who think they already understand window functions.

*New here? See the [Course Overview](<00 - Course Overview.ipynb>) for how to actually run these notebooks, viewing this on GitHub shows a preview, it won't execute the SQL.*

> **By the end of this notebook you'll be able to:** control exactly which rows a window function sees with `ROWS BETWEEN`, split ranked rows into equal buckets with `NTILE()`, measure relative standing with `PERCENT_RANK()` and `CUME_DIST()`, and explain why `LAST_VALUE()` doesn't do what it looks like it does until you widen its frame.

## 1. Frames: controlling exactly which rows a window function sees

Every window function runs over a **frame**, the specific set of rows, within the current `PARTITION BY` group, that it actually looks at. `05`'s running total used the default frame without naming it: when `ORDER BY` is present inside `OVER(...)` and no frame is specified, SQL Server defaults to `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`, everything from the start of the partition up to and including the current row. That's exactly what makes a running total a *running* total.

`ROWS BETWEEN` lets you name a different frame explicitly, not just "everything so far," but a fixed number of rows before and after the current one. That turns a running total into something like a moving average.

**Example:**

A 3-order moving average of `TotalDue`, the current order plus the one immediately before and after it, within each customer's own order history:

In [ ]:
SELECT
    CustomerID,
    SalesOrderID,
    OrderDate,
    TotalDue,
    AVG(TotalDue) OVER (
        PARTITION BY CustomerID
        ORDER BY OrderDate
        ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING
    ) AS "3-Order Moving Average"
FROM Sales.SalesOrderHeader
ORDER BY CustomerID, OrderDate;
-- Each row's average includes at most 3 orders: itself, the one before, the one after.
-- The first and last order in each customer's history only have 2 neighbors to average, not 3.

## 2. NTILE(): splitting ranked rows into equal buckets

`NTILE(n)` divides the rows in a partition, in the order given by `ORDER BY`, into `n` roughly equal-sized buckets, numbered `1` through `n`. It's the function behind "top quartile," "bottom decile," and similar bucketed reporting.

**Example:**

Split every customer into 4 spending quartiles, based on how much they've spent in total. Quartile 1 is the top spenders, because `ORDER BY "Total Spent" DESC` puts the highest values first:

In [ ]:
WITH customer_totals AS (
    SELECT
        CustomerID,
        SUM(TotalDue) AS "Total Spent"
    FROM Sales.SalesOrderHeader
    GROUP BY CustomerID
)
SELECT
    CustomerID,
    "Total Spent",
    NTILE(4) OVER (ORDER BY "Total Spent" DESC) AS "Spending Quartile"
FROM customer_totals
ORDER BY "Total Spent" DESC;
-- If the customer count doesn't divide evenly by 4, the earlier buckets absorb the extra rows,
-- NTILE doesn't leave a partial 5th bucket, it keeps every bucket within 1 row of the others

## 3. PERCENT_RANK() and CUME_DIST(): relative standing, not a fixed bucket count

`NTILE()` forces rows into a number of buckets you choose. `PERCENT_RANK()` and `CUME_DIST()` instead answer "where does this row sit, as a percentage," without picking a bucket count up front, both range from 0 to 1.

They're not the same percentage, though:

- `PERCENT_RANK()` is `(rank - 1) / (total rows - 1)`. The lowest-ranked row in the partition is always exactly `0`, the highest is always exactly `1`.
- `CUME_DIST()` is `(rows at or below this value) / (total rows)`. The lowest row isn't necessarily `0` (there's at least one row at or below it: itself), and any row tied for the maximum is exactly `1`.

**Example:**

In [ ]:
WITH customer_totals AS (
    SELECT
        CustomerID,
        SUM(TotalDue) AS "Total Spent"
    FROM Sales.SalesOrderHeader
    GROUP BY CustomerID
)
SELECT
    CustomerID,
    "Total Spent",
    PERCENT_RANK() OVER (ORDER BY "Total Spent") AS "Percent Rank",
    CUME_DIST()    OVER (ORDER BY "Total Spent") AS "Cumulative Distribution"
FROM customer_totals
ORDER BY "Total Spent" DESC;
-- The single lowest spender: Percent Rank = 0 exactly. Cumulative Distribution won't be 0,
-- there's always at least one row (itself) at or below its own value

## 4. The LAST_VALUE() frame trap

`FIRST_VALUE()` and `LAST_VALUE()` sound symmetrical, and `FIRST_VALUE()` behaves the way you'd expect with the default frame. `LAST_VALUE()` doesn't, and this is one of the most common SQL Server window function mistakes, code that runs without error and looks plausible, but is quietly wrong.

Remember the default frame from section 1: `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`. "Up to and including the current row" is exactly right for a running total. It's exactly wrong for `LAST_VALUE()`, because it means every row's own frame ends *at itself*. Ask `LAST_VALUE()` for "the last value in this frame," and the frame never extends past the current row, so it hands back the current row, every single time.

In [ ]:
SELECT
    CustomerID,
    SalesOrderID,
    OrderDate,
    TotalDue,
    LAST_VALUE(TotalDue) OVER (
        PARTITION BY CustomerID
        ORDER BY OrderDate
    ) AS "'Last' Order Total (looks right, isn't)"
FROM Sales.SalesOrderHeader
ORDER BY CustomerID, OrderDate;
-- Run this and compare the new column to TotalDue on the same row: they match, every time.
-- That's the bug. This is supposed to be the customer's MOST RECENT order total on every row,
-- not each row's own total repeated back at it

Here's what's actually happening, side by side:

<p align="center">
  <img src="graphics/06_last_value_frame_trap.png" width="600" alt="Default RANGE frame stops at the current row, so LAST_VALUE returns the current row's own value. Widening to ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING makes it return the true last row in the partition.">
</p>

The fix is the same tool from section 1, `ROWS BETWEEN`, just pushed all the way to both edges of the partition instead of a few neighboring rows:

In [ ]:
SELECT
    CustomerID,
    SalesOrderID,
    OrderDate,
    TotalDue,
    LAST_VALUE(TotalDue) OVER (
        PARTITION BY CustomerID
        ORDER BY OrderDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS "True Last Order Total"
FROM Sales.SalesOrderHeader
ORDER BY CustomerID, OrderDate;
-- ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING widens the frame to the WHOLE
-- partition for every row, so LAST_VALUE now sees all the way to each customer's actual
-- most recent order, not just up to itself

**The one-line version to remember:** if `LAST_VALUE()` (or you're relying on `FIRST_VALUE()` and `LAST_VALUE()` agreeing with each other) is returning something suspicious, check the frame before you check anything else, it's almost always the default `RANGE ... CURRENT ROW` quietly limiting what "last" can even mean.

## What's Next

You can now shape exactly which rows a window function sees with `ROWS BETWEEN`, bucket ranked rows with `NTILE()`, measure relative standing with `PERCENT_RANK()` and `CUME_DIST()`, and you've seen why `LAST_VALUE()` needs an explicit frame to do what its name implies. Next: `08a - Changing Data, Part 1`, where you move from reading data to writing it.

---

[← 05 - Window Functions, Part 1](<05 - Window Functions, Part 1.ipynb>) · [Course Overview](<00 - Course Overview.ipynb>) · [08a - Changing Data, Part 1 →](<08a - Changing Data, Part 1 - INSERT, UPDATE, DELETE.ipynb>)

*SQL_Tutorial* is written and maintained by Samuel Shaibu as part of *All About Data & More*. Licensed under [MIT](LICENSE).